In [43]:
#gradio ve pandas kütüphanelerinin yüklenmesi
pip install pandas gradio

In [1]:
import gradio as gr
import pandas as pd
import os

class BinaRiskAnalizi:
    def __init__(self):
        # İlçe risk dereceleri
        self.ilce_risk_derece = {
            "Adalar": 1, "Arnavutköy": 3, "Ataşehir": 1, "Avcılar": 1, "Bağcılar": 2,
            "Bahçelievler": 1, "Bakırköy": 1, "Başakşehir": 2, "Bayrampaşa": 2, 
            "Beşiktaş": 2, "Beykoz": 2, "Beylikdüzü": 1, "Beyoğlu": 2, "Büyükçekmece": 2, 
            "Çatalca": 3, "Çekmeköy": 2, "Esenler": 2, "Esenyurt": 2, "Eyüp": 3, 
            "Fatih": 1, "Gaziosmanpaşa": 2, "Güngören": 2, "Kadıköy": 1, "Kağıthane": 3, 
            "Kartal": 1, "Küçükçekmece": 1, "Maltepe": 1, "Pendik": 1, "Sancaktepe": 1, 
            "Sarıyer": 3, "Şile": 2, "Silivri": 3, "Şişli": 2, "Sultanbeyli": 1, 
            "Sultangazi": 2, "Tuzla": 1, "Ümraniye": 1, "Üsküdar": 1, "Zeytinburnu": 1
        }
        
        # Ağırlıklar
        self.konum_agirlik = 0.40
        self.yas_agirlik = 0.30
        self.yapi_agirlik = 0.20
        self.kat_sayisi_agirlik = 0.10
        self.bina_guclendirmesi_agirlik = 0.20
        self.kesik_kolon_agirlik = 0.20
        self.yangin_gecmisi_agirlik = 0.20
        self.zemin_isyeri_agirlik = 0.20
        self.kacak_yapi_agirlik = 0.20

    def risk_puani_hesapla(self, konum, yas, yapi, kat_sayisi, bina_guclendirmesi, kesik_kolon, yangin_gecmisi, zemin_isyeri, kacak_yapi):
        # Konum puanı hesaplama
        konum = self.ilce_risk_derece[konum]
        konum_puan = {1: 100, 2: 75, 3: 50, 4: 25}.get(konum)

        # Binanın yaşı puanı hesaplama
        yas_puan = {'0-10': 20, '11-20': 50, '21-30': 80}.get(yas, 100)

        # Yapı tipi puanı hesaplama
        yapi_puan = {'çelik': 15, 'betonarme': 50, 'ahşap': 90, 'yığma': 100}.get(yapi.lower())

        # Kat sayısı puanı hesaplama
        kat_sayisi_puan = {'0-3': 20, '4-8': 50, '9-20': 100}.get(kat_sayisi, 25)

        # Ana risk puanı hesaplama
        ana_risk_puan = (konum_puan * self.konum_agirlik +
                         yas_puan * self.yas_agirlik +
                         yapi_puan * self.yapi_agirlik +
                         kat_sayisi_puan * self.kat_sayisi_agirlik)

        # Yardımcı risk faktörlerinin puan hesaplaması
        bina_guclendirmesi_puan = -20 if bina_guclendirmesi else 0
        kesik_kolon_puan = 50 if kesik_kolon else 0
        yangin_gecmisi_puan = 20 if yangin_gecmisi else 0
        zemin_isyeri_puan = 10 if zemin_isyeri else 0
        kacak_yapi_puan = 70 if kacak_yapi else 0

        # Yardımcı risk puanı hesaplama
        yardimci_risk_puan = (bina_guclendirmesi_puan * self.bina_guclendirmesi_agirlik +
                              kesik_kolon_puan * self.kesik_kolon_agirlik +
                              yangin_gecmisi_puan * self.yangin_gecmisi_agirlik +
                              zemin_isyeri_puan * self.zemin_isyeri_agirlik +
                              kacak_yapi_puan * self.kacak_yapi_agirlik)

        # Toplam risk puanı hesaplama
        toplam_risk_puan = ana_risk_puan * 0.80 + yardimci_risk_puan * 0.20

        return toplam_risk_puan

    def risk_seviyesi(self, puan):
        if puan < 45:
            return "Düşük Riskli Bina"
        elif puan < 67:
            return "Orta Riskli Bina"
        elif puan <= 100:
            return "Yüksek Riskli Bina"

    def veriyi_kaydet(self, data): #verileri yerel konumda bir excel dosyasına kaydetme
        directory = "Samsung_IC"
        file_path = os.path.join(directory, "bina_risk_verileri.xlsx")
        
        if not os.path.exists(directory):
            os.makedirs(directory)

        df = pd.DataFrame([data])
        if os.path.exists(file_path):
            df_existing = pd.read_excel(file_path)
            df = pd.concat([df_existing, df], ignore_index=True)
        df.to_excel(file_path, index=False)

    def hesapla(self, konum, yas, yapi, kat_sayisi, bina_guclendirmesi, kesik_kolon, yangin_gecmisi, zemin_isyeri, kacak_yapi, deprem_sigorta, police_no):
        # Risk puanını hesaplama
        risk_puani = self.risk_puani_hesapla(konum, yas, yapi, kat_sayisi, bina_guclendirmesi, kesik_kolon, yangin_gecmisi, zemin_isyeri, kacak_yapi)
        seviye = self.risk_seviyesi(risk_puani)
        
        # Sonucu ekranda gösterme
        result_text = f"Toplam Risk Puanı: {risk_puani:.2f} ({seviye})"
        
        data = {
            "Konum": konum,
            "Yaş": yas,
            "Yapı Tipi": yapi,
            "Kat Sayısı": kat_sayisi,
            "Bina Güçlendirmesi": bina_guclendirmesi,
            "Kesik Kolon": kesik_kolon,
            "Yangın Geçmişi": yangin_gecmisi,
            "Zeminde İşyeri": zemin_isyeri,
            "Kaçak Yapı": kacak_yapi,
            "Deprem Sigortası": deprem_sigorta,
            "Poliçe Numarası": police_no if deprem_sigorta else "Yok",
            "Risk Puanı": risk_puani,
            "Risk Seviyesi": seviye
        }
        
        self.veriyi_kaydet(data)
        return result_text



# Gradio arayüzü
bina_analiz = BinaRiskAnalizi()

def police_girisi(deprem_sigorta): #Eğer deprem sigortası butonu aktifleşirse poliçe numarası giriş kutucuğu aktifleştirilir
    return gr.update(visible=deprem_sigorta)

with gr.Blocks() as iface:
    konum = gr.Dropdown(bina_analiz.ilce_risk_derece.keys(), label="Konum")
    yas = gr.Dropdown(["0-10", "11-20", "21-30", "30+"], label="Binanın Yaşı")
    yapi = gr.Dropdown(["Çelik", "Betonarme", "Ahşap", "Yığma"], label="Yapı Tipi")
    kat_sayisi = gr.Dropdown(["0-3", "4-8", "9-20", "20+"], label="Kat Sayısı")
    bina_guclendirmesi = gr.Checkbox(label="Bina güçlendirmesi var mı?")
    kesik_kolon = gr.Checkbox(label="Kesik kolon var mı?")
    yangin_gecmisi = gr.Checkbox(label="Yangın geçmişi var mı?")
    zemin_isyeri = gr.Checkbox(label="Zeminde işyeri var mı?")
    kacak_yapi = gr.Checkbox(label="Bina kaçak yapı mı?")
    deprem_sigorta = gr.Checkbox(label="Deprem sigortanız var mı?")
    police_no = gr.Textbox(placeholder="Poliçe Numarasını Giriniz. İstemiyorsanız Boş Bırakabilirsiniz.", label="Poliçe Numarası", visible=False)
    
    deprem_sigorta.change(
        lambda x: police_girisi(x),
        inputs=[deprem_sigorta],
        outputs=[police_no]
    )

    hesapla_btn = gr.Button("Hesapla")
    output_text = gr.Textbox(label="Sonuç")

    hesapla_btn.click(
        bina_analiz.hesapla,
        inputs=[konum, yas, yapi, kat_sayisi, bina_guclendirmesi, kesik_kolon, yangin_gecmisi, zemin_isyeri, kacak_yapi, deprem_sigorta, police_no],
        outputs=output_text
    )

iface.launch(share=True)

Running on local URL:  http://127.0.0.1:7860
IMPORTANT: You are using gradio version 4.15.0, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://02478f51d7f94d368e.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
